In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

BASE_PATH = "/content/drive/MyDrive/AutoSOC_Project"

files = [
    1,
    9,
    16,
    18
]

for i in files:

    path = os.path.join(
        BASE_PATH,
        f"GroundTruth_Network_{i}.csv"
    )

    df = pd.read_csv(
        path,
        usecols=["ts","src_ip","dst_ip","type"],
        low_memory=False
    )

    print("\n======================")
    print(f"GroundTruth_Network_{i}")
    print("======================")

    print("Shape:", df.shape)

    print("\nAttack type:")
    print(df["type"].value_counts())

    print("\nTime range:")
    print(
        df["ts"].min(),
        "to",
        df["ts"].max()
    )

    print("\nTop source IP:")
    print(
        df["src_ip"].value_counts().head(3)
    )

Mounted at /content/drive

GroundTruth_Network_1
Shape: (1000000, 4)

Attack type:
type
scanning    1000000
Name: count, dtype: int64

Time range:
1556021410 to 1556025857

Top source IP:
src_ip
192.168.1.30    536244
192.168.1.32    342992
192.168.1.31     71862
Name: count, dtype: int64

GroundTruth_Network_9
Shape: (1000000, 4)

Attack type:
type
ddos    1000000
Name: count, dtype: int64

Time range:
1556206297 to 1556218827

Top source IP:
src_ip
192.168.1.30    522430
192.168.1.31    475567
192.168.1.38      1168
Name: count, dtype: int64

GroundTruth_Network_16
Shape: (1000000, 4)

Attack type:
type
password    534871
xss         465129
Name: count, dtype: int64

Time range:
1556323577 to 1556349995

Top source IP:
src_ip
192.168.1.32    497399
192.168.1.35    170635
192.168.1.36    131623
Name: count, dtype: int64

GroundTruth_Network_18
Shape: (801188, 4)

Attack type:
type
xss           605685
backdoor      188864
ransomware      5629
mitm            1010
Name: count, dtype: i

In [2]:
import pandas as pd
import os

BASE_PATH = "/content/drive/MyDrive/AutoSOC_Project"

files = [1, 9, 16, 18]

all_events = []

for i in files:

    path = os.path.join(
        BASE_PATH,
        f"GroundTruth_Network_{i}.csv"
    )

    df = pd.read_csv(
        path,
        usecols=[
            "ts",
            "src_ip",
            "dst_ip",
            "src_port",
            "dst_port",
            "proto",
            "type"
        ],
        low_memory=False
    )

    df["source_file"] = f"GroundTruth_{i}"

    # lowercase
    df["type"] = (
        df["type"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    all_events.append(df)

    print(
        f"Loaded GroundTruth_{i}:",
        df.shape
    )


# Merge all attack events

events = pd.concat(
    all_events,
    ignore_index=True
)


# Sort by attacker and time

events = events.sort_values(
    [
        "src_ip",
        "ts"
    ]
).reset_index(drop=True)


print("\nTotal events:")
print(events.shape)


print("\nAttack distribution:")
print(
    events["type"].value_counts()
)


print("\nExample timeline:")
display(
    events.head(20)
)

Loaded GroundTruth_1: (1000000, 8)
Loaded GroundTruth_9: (1000000, 8)
Loaded GroundTruth_16: (1000000, 8)
Loaded GroundTruth_18: (801188, 8)

Total events:
(3801188, 8)

Attack distribution:
type
xss           1070814
ddos          1000000
scanning      1000000
password       534871
backdoor       188864
ransomware       5629
mitm             1010
Name: count, dtype: int64

Example timeline:


,ts,src_ip,src_port,dst_ip,dst_port,proto,type,source_file
0,1556025489,103.13.80.89,3,192.168.1.30,0,icmp,scanning,GroundTruth_1
1,1556025606,103.13.80.89,3,192.168.1.32,0,icmp,scanning,GroundTruth_1
2,1556423393,13.55.50.68,123,192.168.1.37,52589,udp,ransomware,GroundTruth_18
3,1556423457,13.55.50.68,123,192.168.1.37,49842,udp,ransomware,GroundTruth_18
4,1556423586,13.55.50.68,123,192.168.1.37,36542,udp,ransomware,GroundTruth_18
5,1556423842,13.55.50.68,123,192.168.1.37,56197,udp,ransomware,GroundTruth_18
6,1556424354,13.55.50.68,123,192.168.1.37,40653,udp,ransomware,GroundTruth_18
7,1556425378,13.55.50.68,123,192.168.1.37,53875,udp,ransomware,GroundTruth_18
8,1556427427,13.55.50.68,123,192.168.1.37,42774,udp,ransomware,GroundTruth_18
9,1556025740,169.254.130.193,23338,192.168.1.30,10900,udp,scanning,GroundTruth_1


In [3]:
# ==========================================
# MULTI-STAGE ATTACK CAMPAIGN CORRELATION
# ==========================================

# Sort by attacker and time
events = events.sort_values(
    [
        "src_ip",
        "ts"
    ]
).reset_index(drop=True)


# Time gap between consecutive attacks
events["time_gap"] = (
    events
    .groupby("src_ip")["ts"]
    .diff()
)


# If same attacker has events within 1 day
# consider them part of same campaign

CAMPAIGN_WINDOW = 86400   # 24 hours


events["new_campaign"] = (
    events["time_gap"].isna()
    |
    (events["time_gap"] > CAMPAIGN_WINDOW)
)


# Create campaign ID

events["campaign_id"] = (
    events
    .groupby("src_ip")["new_campaign"]
    .cumsum()
)


# Create attack sequence

def get_sequence(x):

    result = []

    for attack in x:

        if attack not in result:
            result.append(attack)

    return " → ".join(result)



campaigns = (
    events
    .groupby(
        [
            "src_ip",
            "campaign_id"
        ]
    )
    .agg(
        start_time=("ts","min"),
        end_time=("ts","max"),
        total_events=("ts","count"),
        unique_targets=("dst_ip","nunique"),
        attack_stages=("type","nunique"),
        attack_sequence=("type",get_sequence)
    )
    .reset_index()
)


# Sort most interesting first

campaigns = campaigns.sort_values(
    [
        "attack_stages",
        "total_events"
    ],
    ascending=False
)


print("Total campaigns:",
      len(campaigns))


print("\nTop Multi-stage Campaigns:")

display(
    campaigns.head(20)
)

Total campaigns: 46

Top Multi-stage Campaigns:


,src_ip,campaign_id,start_time,end_time,total_events,unique_targets,attack_stages,attack_sequence
37,192.168.1.35,1,1556334041,1556368853,505481,26,2,password → xss
32,192.168.1.32,2,1556323577,1556345172,497399,36,2,password → xss
21,192.168.1.193,2,1556424140,1556531133,189467,2,2,ransomware → backdoor
34,192.168.1.33,2,1556423390,1556530505,3055,9,2,ransomware → backdoor
39,192.168.1.37,1,1556423390,1556529539,1953,9,2,ransomware → backdoor
26,192.168.1.30,1,1556021410,1556025857,536244,244,1,scanning
27,192.168.1.30,2,1556206298,1556218827,522430,6,1,ddos
29,192.168.1.31,2,1556206297,1556218827,475567,15,1,ddos
31,192.168.1.32,1,1556021410,1556025857,342992,244,1,scanning
42,192.168.1.39,1,1556348165,1556368853,341001,23,1,xss


In [4]:
# ============================================================
# FINAL AUTOSOC:
# Risk Scoring
# MITRE Mapping
# Incident Investigation
# Response Recommendation
# ============================================================

import numpy as np
import pandas as pd


# -------------------------------
# 1. Attack Severity Knowledge
# -------------------------------

severity_map = {

    "scanning": 25,
    "password": 40,
    "xss": 45,
    "ddos": 70,
    "dos": 65,
    "backdoor": 75,
    "ransomware": 90,
    "mitm": 80,
    "injection": 60
}


# -------------------------------
# 2. MITRE ATT&CK Mapping
# -------------------------------

mitre_map = {

    "scanning":
    "T1046 - Network Service Scanning",

    "password":
    "T1110 - Brute Force",

    "xss":
    "T1189 - Drive-by Compromise / Web Exploitation",

    "ddos":
    "T1498 - Network Denial of Service",

    "dos":
    "T1499 - Endpoint Denial of Service",

    "backdoor":
    "T1059 - Command and Scripting Interpreter",

    "ransomware":
    "T1486 - Data Encrypted for Impact",

    "mitm":
    "T1557 - Adversary-in-the-Middle"
}


# -------------------------------
# 3. Base Severity from Attack Chain
# -------------------------------

def calculate_base_risk(sequence):

    attacks = sequence.split(" → ")

    scores = []

    for attack in attacks:

        attack = attack.strip()

        scores.append(
            severity_map.get(
                attack,
                30
            )
        )

    return max(scores)


campaigns["base_severity"] = (
    campaigns["attack_sequence"]
    .apply(calculate_base_risk)
)


# -------------------------------
# 4. Campaign Intensity Score
# -------------------------------

campaigns["event_score"] = (
    campaigns["total_events"]
    .rank(pct=True)
    * 20
)

campaigns["target_score"] = (
    campaigns["unique_targets"]
    .rank(pct=True)
    * 15
)


# Multi-stage bonus
campaigns["stage_bonus"] = (
    campaigns["attack_stages"] - 1
) * 15


# Final Risk Score

campaigns["risk_score"] = (
    campaigns["base_severity"]
    +
    campaigns["event_score"]
    +
    campaigns["target_score"]
    +
    campaigns["stage_bonus"]
)


campaigns["risk_score"] = (
    campaigns["risk_score"]
    .clip(0,100)
    .round(2)
)


# -------------------------------
# 5. Risk Category
# -------------------------------

def risk_level(score):

    if score >= 80:
        return "CRITICAL"

    elif score >= 60:
        return "HIGH"

    elif score >= 40:
        return "MEDIUM"

    else:
        return "LOW"


campaigns["risk_level"] = (
    campaigns["risk_score"]
    .apply(risk_level)
)


# -------------------------------
# 6. MITRE Technique
# -------------------------------

def get_mitre(sequence):

    attacks = sequence.split(" → ")

    techniques=[]

    for attack in attacks:

        techniques.append(
            mitre_map.get(
                attack.strip(),
                "Unknown"
            )
        )

    return " | ".join(
        techniques
    )


campaigns["mitre_attack"] = (
    campaigns["attack_sequence"]
    .apply(get_mitre)
)


# -------------------------------
# 7. Incident Investigation Summary
# -------------------------------

def make_summary(row):

    return (
        f"Source IP {row['src_ip']} performed "
        f"a {row['attack_sequence']} attack chain. "
        f"The campaign generated "
        f"{row['total_events']} events targeting "
        f"{row['unique_targets']} hosts. "
        f"Risk score: {row['risk_score']}/100 "
        f"({row['risk_level']})."
    )


campaigns["incident_summary"] = (
    campaigns.apply(
        make_summary,
        axis=1
    )
)


# -------------------------------
# 8. Response Recommendation
# -------------------------------

def response(row):

    if row["risk_level"] == "CRITICAL":

        return (
            "Immediately investigate source IP; "
            "isolate affected systems; "
            "preserve forensic logs; "
            "block malicious activity."
        )

    elif row["risk_level"] == "HIGH":

        return (
            "Increase monitoring; "
            "review affected hosts; "
            "investigate attacker behavior."
        )

    else:

        return (
            "Continue monitoring and collect more evidence."
        )


campaigns["response_recommendation"] = (
    campaigns.apply(
        response,
        axis=1
    )
)


# -------------------------------
# FINAL REPORT
# -------------------------------

final_autosoc_report = campaigns[
    [
        "src_ip",
        "campaign_id",
        "attack_sequence",
        "attack_stages",
        "total_events",
        "unique_targets",
        "risk_score",
        "risk_level",
        "mitre_attack",
        "incident_summary",
        "response_recommendation"
    ]
].sort_values(
    "risk_score",
    ascending=False
)


print("FINAL AUTOSOC REPORT ✅")

display(
    final_autosoc_report.head(20)
)


# Save

save_path = (
    "/content/drive/MyDrive/"
    "AutoSOC_Project/"
    "FINAL_AutoSOC_Report.csv"
)


final_autosoc_report.to_csv(
    save_path,
    index=False
)


print("\nSaved:")
print(save_path)

FINAL AUTOSOC REPORT ✅


,src_ip,campaign_id,attack_sequence,attack_stages,total_events,unique_targets,risk_score,risk_level,mitre_attack,incident_summary,response_recommendation
34,192.168.1.33,2,ransomware → backdoor,2,3055,9,100.00,CRITICAL,T1486 - Data Encrypted for Impact | T1059 - Co...,Source IP 192.168.1.33 performed a ransomware ...,Immediately investigate source IP; isolate aff...
21,192.168.1.193,2,ransomware → backdoor,2,189467,2,100.00,CRITICAL,T1486 - Data Encrypted for Impact | T1059 - Co...,Source IP 192.168.1.193 performed a ransomware...,Immediately investigate source IP; isolate aff...
39,192.168.1.37,1,ransomware → backdoor,2,1953,9,100.00,CRITICAL,T1486 - Data Encrypted for Impact | T1059 - Co...,Source IP 192.168.1.37 performed a ransomware ...,Immediately investigate source IP; isolate aff...
27,192.168.1.30,2,ddos,1,522430,6,100.00,CRITICAL,T1498 - Network Denial of Service,Source IP 192.168.1.30 performed a ddos attack...,Immediately investigate source IP; isolate aff...
29,192.168.1.31,2,ddos,1,475567,15,100.00,CRITICAL,T1498 - Network Denial of Service,Source IP 192.168.1.31 performed a ddos attack...,Immediately investigate source IP; isolate aff...
36,192.168.1.34,2,mitm,1,512,90,100.00,CRITICAL,T1557 - Adversary-in-the-Middle,Source IP 192.168.1.34 performed a mitm attack...,Immediately investigate source IP; isolate aff...
30,192.168.1.31,3,mitm,1,489,80,100.00,CRITICAL,T1557 - Adversary-in-the-Middle,Source IP 192.168.1.31 performed a mitm attack...,Immediately investigate source IP; isolate aff...
1,13.55.50.68,1,ransomware,1,7,1,100.00,CRITICAL,T1486 - Data Encrypted for Impact,Source IP 13.55.50.68 performed a ransomware a...,Immediately investigate source IP; isolate aff...
44,203.14.129.10,1,ransomware,1,7,1,100.00,CRITICAL,T1486 - Data Encrypted for Impact,Source IP 203.14.129.10 performed a ransomware...,Immediately investigate source IP; isolate aff...
16,192.168.1.152,2,mitm,1,2,2,92.77,CRITICAL,T1557 - Adversary-in-the-Middle,Source IP 192.168.1.152 performed a mitm attac...,Immediately investigate source IP; isolate aff...



Saved:
/content/drive/MyDrive/AutoSOC_Project/FINAL_AutoSOC_Report.csv
